# Chapter 8 &mdash; Error-Correcting Design I: the RE for Hamming Distance 2

**Concept 6 of the Chapter 8 decomposition:** *Error-Correcting Design I: the RE for "within Hamming Distance 2"*

Enumerate the $\binom{4}{2}=6$ ways of denting `0101`, write `?` as `(0+1)`, and union them.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-Hamming-Distance-RE/Concept-Hamming-Distance-RE.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Design task: accept every 4-bit string **within Hamming distance 2** of `0101`.

The systematic construction: a string at distance **exactly** 2 differs in exactly two
positions, and there are $\binom{4}{2}=6$ choices of which two. For each choice write
the pattern with those positions **denteded** &mdash; a "don't care", written `(0+1)`.

Union the six, and add the distance-1 and distance-0 cases &mdash; or note that
`(0+1)` covers the original symbol too, so the six patterns already include distances
0, 1 and 2.

The count is a sanity check you can compute independently:
$\sum_{k\le2}\binom{4}{k} = 1+4+6 = 11$ strings.

## 2. Definitions

### Build the RE from the position choices

In [ ]:
from itertools import combinations, product
TARGET = '0101'

def dent_re(target, k):
    alts = []
    for pos in combinations(range(len(target)), k):
        alts.append(''.join('(0+1)' if i in pos else target[i]
                            for i in range(len(target))))
    return alts

alts = dent_re(TARGET, 2)
RE2 = '+'.join(alts)
print("%d alternatives, e.g. %s" % (len(alts), alts[0]))

### The reference: Hamming distance

In [ ]:
def ham(a, b): return sum(x != y for x, y in zip(a, b))
def within2(s): return len(s) == len(TARGET) and ham(s, TARGET) <= 2

## 3. Tests

Six alternatives, as $\binom{4}{2}$ predicts.

In [ ]:
from math import comb
print("C(4,2) =", comb(4, 2), " alternatives built :", len(alts))
assert len(alts) == comb(4, 2)

The RE accepts exactly the strings within distance 2.

In [ ]:
def re_dfa(r): return min_dfa(nfa2dfa(re2nfa(r)))
D = re_dfa(RE2)
acc = [''.join(p) for p in product('01', repeat=4) if accepts_dfa(D, ''.join(p))]
print("accepted (%d) :" % len(acc), acc)
assert set(acc) == {''.join(p) for p in product('01', repeat=4)
                    if within2(''.join(p))}

The count matches $\sum_{k\le2}\binom{4}{k} = 11$.

In [ ]:
print("1 + 4 + 6 =", comb(4,0) + comb(4,1) + comb(4,2))
assert len(acc) == comb(4,0) + comb(4,1) + comb(4,2) == 11

Distance 0, 1 and 2 are all present &mdash; the dents subsume the smaller distances.

In [ ]:
from collections import Counter
print("by distance :", dict(sorted(Counter(ham(s, TARGET) for s in acc).items())))
assert TARGET in acc, "distance 0 must be included"
assert max(ham(s, TARGET) for s in acc) == 2

Nothing at distance 3 or 4 sneaks in.

In [ ]:
outside = [''.join(p) for p in product('01', repeat=4)
           if ham(''.join(p), TARGET) > 2]
assert not any(accepts_dfa(D, s) for s in outside)
print("all %d strings at distance 3 or 4 are rejected" % len(outside))

## 4. Animation

The distance-2 acceptor for `0101`.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(re_dfa(RE2), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Build the distance-1 RE. How many alternatives, and how many strings?
2. Generalise `dent_re` to distance $k$ over a 5-bit target. Check the count.
3. Why does `(0+1)` in a dent position also cover the *original* symbol?

In [ ]:
# Your work for the exercises above.